# Assessment of the results using annotated data

In [53]:
# read "annotations.csv"
import pandas as pd
sentence_df = pd.read_excel("data/cleaned_patient_deduplicated_sentences_for_embedding.xlsx")
anno_df = pd.read_csv("data/annotations_for_assessment.csv")
anno_df.tail()

,message_id,sentence,sentence_id,Matched_Sentence_Y_clean,Annotated_Label
7156,msg_5152,Even kijken of dit resultaat boekt.,sent_8374,even kijken of dit resultaat boekt.,A
7157,msg_5152,Hij adviseerde mij om na 3 weken weer op contr...,sent_8375,hij adviseerde mij om na 3 weken weer op contr...,M
7158,msg_5155,"Hallo [PERSOON], Bedankt voor de mededeling.",sent_8376,"hallo [persoon], bedankt voor de info.",A
7159,msg_5155,Als pas bij het volgende infuus wordt geprikt ...,sent_8377,is de operatie wel aangevraagd of wordt dit pa...,M
7160,msg_5155,Is dat niet te lang?,sent_8378,dat gaat dus niet opschieten.,A


In [25]:
# count the unique values in "message_id"
anno_df["message_id"].nunique()

2067

In [26]:
# read the robbert document info
doc_info_df = pd.read_csv("results/robbert/robbert_document_info_reduced.csv")
doc_info_df.head()

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Document_Original
0,Ik ben 2 weken geleden met spoed opgenomen in ...,115,115_longen_cardioloog_hart_longarts,"['longen', 'cardioloog', 'hart', 'longarts', '...",['Ze wilden even voor de zekerheid hart en lon...,longen - cardioloog - hart - longarts - longfo...,0.000000,False,Ik ben 2 weken geleden met spoed opgenomen in ...
1,"Ik kreeg acuut een pijnlijke druk op de borst,...",64,64_gewrichten_handen_vingers_pijn,"['gewrichten', 'handen', 'vingers', 'pijn', 'a...","['- Pijn in gewrichten/peesaanhechtingen.', 'I...",gewrichten - handen - vingers - pijn - arm - l...,0.000000,False,"Ik kreeg acuut een pijnlijke druk op de borst,..."
2,Het begon 1 uur na het avondeten.,52,52_eten_eet_voeding_gegeten,"['eten', 'eet', 'voeding', 'gegeten', 'eetlust...","['Wat mag ik wel/ niet eten?', 'ook na het ete...",eten - eet - voeding - gegeten - eetlust - dit...,0.776607,False,Het begon 1 uur na het avondeten.
3,"Ik had al de hele dag migraine, had dus ook we...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.915956,False,"Ik had al de hele dag migraine, had dus ook we..."
4,"Ik werd heel erg misselijk, braakneigingen, du...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.000000,False,"Ik werd heel erg misselijk, braakneigingen, du..."


In [27]:
# load the robbert topic info
topic_info_df = pd.read_csv("results/robbert/robbert_full_topic_info_translated.csv")
print(len(topic_info_df))
topic_info_df.head()

157


,Topic,Count,Name,Representation,Representative_Docs,Rank,Translation,Category
0,32,2868,32_vriendelijke_groet_groeten_vriendelijk,"['vriendelijke', 'groet', 'groeten', 'vriendel...","['Met vriendelijke groet Met', 'met vrie...",1,"['kind', 'regards', 'greetings', 'friendly', '...",Non-Medical
1,0,1484,0_ontlasting_ingeleverd_inleveren_bloed,"['ontlasting', 'ingeleverd', 'inleveren', 'blo...","['Beste, dinsdag 8 maart heb ik bloed laten pr...",2,"['stool sample', 'submitted', 'submit', 'blood...",Medical
2,1,1315,1_bellen_bereikbaar_telefonisch_gebeld,"['bellen', 'bereikbaar', 'telefonisch', 'gebel...","['Hallo, Zouden jullie mij kunnen bellen op ...",3,"['call', 'reachable', 'by phone', 'called', 'c...",Non-Medical
3,2,1292,2_recept_apotheek_sturen_nieuw,"['recept', 'apotheek', 'sturen', 'nieuw', 'fax...","['Graag het recept naar apotheek .', 'Er was ...",4,"['prescription', 'pharmacy', 'send', 'location...",Non-Medical
4,3,1086,3_medicatie_medicijnen_medicijn_bijwerkingen,"['medicatie', 'medicijnen', 'medicijn', 'bijwe...","['Mijn medicatie is bijna op.', 'Zou dit iets ...",5,"['medication', 'medicines', 'medicine', 'side ...",Medical


In [73]:
# show rows where Topic = 108, 144, 149
topic_info_df[topic_info_df["Topic"].isin([108, 144, 149])]

,Topic,Count,Name,Representation,Representative_Docs,Rank,Translation,Category
67,108,233,108_vriendelijke_groet_alvast_geb,"['vriendelijke', 'groet', 'alvast', 'geb', 'hg...","['Met vriendelijke groet, ', 'Met vriende...",68,"['kind', 'regards', 'already', 'gb', 'hgl', 'g...",Non-Medical
135,149,102,149_reactie_antwoord_bedankt_dank,"['reactie', 'antwoord', 'bedankt', 'dank', 'ho...","['Hoi , Bedankt voor de reactie!', 'Hoi , Be...",136,"['response', 'answer', 'thank_you', 'thanks', ...",Non-Medical
147,144,84,144_mvg_swinkels_kollar_alvast,"['mvg', 'swinkels', 'kollar', 'alvast', 'hoor'...","['Mvg ', 'Mvg, ( )', 'Mvg ']",148,"['mvg', 'swinkels', 'kollar', 'already', 'hear...",Non-Medical


In [39]:
# join the rows [0: 7160] of "anno_df" with "doc_info_df" on "sentence" in "anno_df" and "Document" in "doc_info_df"
merged_df = pd.merge(anno_df.iloc[0:7160], doc_info_df.iloc[0:7160], left_on="sentence", right_on="Document")
print(len(merged_df))
merged_df.tail()

5112


,message_id,sentence,sentence_id,Matched_Sentence_Y_clean,Annotated_Label,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Document_Original
5107,msg_5150,Ik heb hierover nog contact met mijn huisarts.,sent_8370,ik heb hierover nog contact met mijn huisarts.,A,Ik heb hierover nog contact met mijn huisarts.,5,5_huisarts_dokter_arts_contact,"['huisarts', 'dokter', 'arts', 'contact', 'med...","['Zou ik naar de huisarts moeten?', 'Ik ben af...",huisarts - dokter - arts - contact - medische ...,0.807997,False,Ik heb hierover nog contact met mijn huisarts.
5108,msg_5152,Mijn huisarts kan de misselijkheid niet verkla...,sent_8372,mijn huisarts kan de misselijkheid niet verkla...,M,Mijn huisarts kan de misselijkheid niet verkla...,132,132_misselijk_misselijkheid_overgeven_eetlust,"['misselijk', 'misselijkheid', 'overgeven', 'e...",['Ik heb daar vannacht van moeten overgeven om...,misselijk - misselijkheid - overgeven - eetlus...,0.000000,False,Mijn huisarts kan de misselijkheid niet verkla...
5109,msg_5152,Hij heeft mijn Omeprazol dosis verhoogd van 1x...,sent_8373,hij heeft mijn omeprazol dosis verhoogd van 1x...,M,Hij heeft mijn Omeprazol dosis verhoogd van 1x...,97,97_esomeprazol_omeprazol_dgs_daags,"['esomeprazol', 'omeprazol', 'dgs', 'daags', '...",['Esomeprazol heb ik voorgeschreven gekregen d...,esomeprazol - omeprazol - dgs - daags - pantop...,0.537006,False,Hij heeft mijn Omeprazol dosis verhoogd van 1x...
5110,msg_5152,Even kijken of dit resultaat boekt.,sent_8374,even kijken of dit resultaat boekt.,A,Even kijken of dit resultaat boekt.,16,16_uitslag_resultaten_hiervan_bekend,"['uitslag', 'resultaten', 'hiervan', 'bekend',...","['Heb je voor mij de uitslag?', 'Hebben jullie...",uitslag - resultaten - hiervan - bekend - resu...,0.680639,False,Even kijken of dit resultaat boekt.
5111,msg_5155,Als pas bij het volgende infuus wordt geprikt ...,sent_8377,is de operatie wel aangevraagd of wordt dit pa...,M,Als pas bij het volgende infuus wordt geprikt ...,15,15_infuus_volgende_infuuscentrum_entyvio,"['infuus', 'volgende', 'infuuscentrum', 'entyv...","['Ik heb 9 augustus wel mijn infuus gehad.', '...",infuus - volgende - infuuscentrum - entyvio - ...,0.874423,False,Als pas bij het volgende infuus wordt geprikt ...


In [75]:
# create a map of topic labels by counting the occurences of "Mapped_Label" in each "Topic" in merged_df
topic_label_map = merged_df.groupby("Topic")["Annotated_Label"].value_counts().unstack(fill_value=0)
# create a new column "Most_Common_Label" in topic_label_map that contains the most common label for each topic
topic_label_map["Most_Common_Label"] = topic_label_map.idxmax(axis=1)

# add Topic 108, 144, 149 to topic_label_map with "Most_Common_Label" as "A"
for topic in [108, 144, 149]:
    topic_label_map.loc[topic] = [0] * (len(topic_label_map.columns) - 1) + ["A"]

topic_label_map.head()

Annotated_Label,A,M,Most_Common_Label
Topic,,,
0,151,16,A
1,114,0,A
2,120,12,A
3,34,128,M
4,102,6,A


In [76]:
# Check whether all topic IDs from 0 to 156 exist

expected_topics = set(range(157))

# If Topic is numeric
actual_topics = set(topic_label_map.index)

missing_topics = expected_topics - actual_topics
extra_topics = actual_topics - expected_topics

print("Total topics found:", len(actual_topics))
print("Missing topics:", sorted(missing_topics))
print("Extra topics:", sorted(extra_topics))

Total topics found: 157
Missing topics: []
Extra topics: []


In [77]:
# map the "mapped_label" in "topic_info_df" to the "Most_Common_Label" in "topic_label_map" based on the "Topic" column
merged_df["Mapped_Label"] = merged_df["Topic"].map(topic_label_map["Most_Common_Label"])
merged_df.head(20)


,message_id,sentence,sentence_id,Matched_Sentence_Y_clean,Annotated_Label,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Document_Original,Mapped_Label
0,msg_4,"Ik kreeg acuut een pijnlijke druk op de borst,...",sent_2,"ik kreeg acuut een pijnlijke druk op de borst,...",M,"Ik kreeg acuut een pijnlijke druk op de borst,...",64,64_gewrichten_handen_vingers_pijn,"['gewrichten', 'handen', 'vingers', 'pijn', 'a...","['- Pijn in gewrichten/peesaanhechtingen.', 'I...",gewrichten - handen - vingers - pijn - arm - l...,0.000000,False,"Ik kreeg acuut een pijnlijke druk op de borst,...",M
1,msg_4,Het begon 1 uur na het avondeten.,sent_3,het begon 1 uur na het avondeten.,A,Het begon 1 uur na het avondeten.,52,52_eten_eet_voeding_gegeten,"['eten', 'eet', 'voeding', 'gegeten', 'eetlust...","['Wat mag ik wel/ niet eten?', 'ook na het ete...",eten - eet - voeding - gegeten - eetlust - dit...,0.776607,False,Het begon 1 uur na het avondeten.,M
2,msg_4,"Ik had al de hele dag migraine, had dus ook we...",sent_4,"ik had al de hele dag migraine, had dus ook we...",M,"Ik had al de hele dag migraine, had dus ook we...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.915956,False,"Ik had al de hele dag migraine, had dus ook we...",M
3,msg_4,"Ik werd heel erg misselijk, braakneigingen, du...",sent_5,"ik werd heel erg misselijk, braakneigingen, du...",M,"Ik werd heel erg misselijk, braakneigingen, du...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.000000,False,"Ik werd heel erg misselijk, braakneigingen, du...",M
4,msg_4,Op de eerste hulp hebben ze het hart de longen...,sent_6,op de eerste hulp hebben ze het hart de longen...,M,Op de eerste hulp hebben ze het hart de longen...,115,115_longen_cardioloog_hart_longarts,"['longen', 'cardioloog', 'hart', 'longarts', '...",['Ze wilden even voor de zekerheid hart en lon...,longen - cardioloog - hart - longarts - longfo...,0.622709,False,Op de eerste hulp hebben ze het hart de longen...,M
5,msg_4,hier kwamen geen rare dingen uit.,sent_7,hier kwamen geen rare dingen uit.,A,hier kwamen geen rare dingen uit.,153,153_bekend_uitgekomen_dingen_onbekend,"['bekend', 'uitgekomen', 'dingen', 'onbekend',...","['Is hier inmiddels al iets van bekend ?', 'Is...",bekend - uitgekomen - dingen - onbekend - nieu...,1.000000,False,hier kwamen geen rare dingen uit.,A
6,msg_4,Later op de dag bij het eten van een beschuitj...,sent_8,later op de dag bij het eten van een beschuitj...,M,Later op de dag bij het eten van een beschuitj...,52,52_eten_eet_voeding_gegeten,"['eten', 'eet', 'voeding', 'gegeten', 'eetlust...","['Wat mag ik wel/ niet eten?', 'ook na het ete...",eten - eet - voeding - gegeten - eetlust - dit...,0.563863,False,Later op de dag bij het eten van een beschuitj...,M
7,msg_4,Ze hebben een echo van de gal gemaakt ook zond...,sent_9,ze hebben een echo van de gal gemaakt ook zond...,M,Ze hebben een echo van de gal gemaakt ook zond...,0,0_ontlasting_ingeleverd_inleveren_bloed,"['ontlasting', 'ingeleverd', 'inleveren', 'blo...","['Beste, dinsdag 8 maart heb ik bloed laten pr...",ontlasting - ingeleverd - inleveren - bloed - ...,0.000000,False,Ze hebben een echo van de gal gemaakt ook zond...,A
8,msg_4,Er is nog een aanvullend onderzoek van de slok...,sent_10,er is nog een aanvullend onderzoek van de slok...,M,Er is nog een aanvullend onderzoek van de slok...,43,43_darmonderzoek_darm_maagonderzoek_darmkanker,"['darmonderzoek', 'darm', 'maagonderzoek', 'da...","['Beste, Afgelopen vrijdag heb ik een darmon...",darmonderzoek - darm - maagonderzoek - darmkan...,0.547418,False,Er is nog een aanvullend onderzoek van de slok...,M
9,msg_4,"Ik heb nog altijd last van oprispinge

In [78]:
# calculate the accuracy of the "Mapped_Label" compared to the "Annotated_Label" in "merged_df"
accuracy = (merged_df["Mapped_Label"] == merged_df["Annotated_Label"]).mean()
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.83


In [79]:
# calculate the occurences of each "Mapped_Label" in "merged_df" grouped by "message_id"
mapped_label_counts = merged_df.groupby("message_id")["Mapped_Label"].value_counts().unstack(fill_value=0)
# calculate the percentage of messages that have 0 "Mapped_Label" values that are "M"
percentage_no_m = (mapped_label_counts["M"] == 0).mean()
print(f"Percentage of messages with no 'M' label: {percentage_no_m:.2f}")
# calculate the occurences of each "Annotated_Label" in "merged_df" grouped by "message_id"
annotated_label_counts = merged_df.groupby("message_id")["Annotated_Label"].value_counts().unstack(fill_value=0)
# calculate the percentage of messages that have 0 "Annotated_Label" values that are "M"
percentage_no_m_annotated = (annotated_label_counts["M"] == 0).mean()
print(f"Percentage of messages with no 'M' label in annotated labels: {percentage_no_m_annotated:.2f}")

Percentage of messages with no 'M' label: 0.42
Percentage of messages with no 'M' label in annotated labels: 0.51


In [80]:
# map the "mapped_label" in "doc_info_df" to the "Most_Common_Label" in "topic_label_map" based on the "Topic" column
doc_info_df["Mapped_Label"] = doc_info_df["Topic"].map(topic_label_map["Most_Common_Label"])
# merge doc_info_df with sentence_df on index
merged_doc_sentence_df = pd.merge(doc_info_df, sentence_df, left_index=True, right_index=True)
merged_doc_sentence_df.head()


,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,Document_Original,Mapped_Label,message_id,sentence,sentence_id
0,Ik ben 2 weken geleden met spoed opgenomen in ...,115,115_longen_cardioloog_hart_longarts,"['longen', 'cardioloog', 'hart', 'longarts', '...",['Ze wilden even voor de zekerheid hart en lon...,longen - cardioloog - hart - longarts - longfo...,0.000000,False,Ik ben 2 weken geleden met spoed opgenomen in ...,M,msg_4,Ik ben 2 weken geleden met spoed opgenomen in ...,sent_1
1,"Ik kreeg acuut een pijnlijke druk op de borst,...",64,64_gewrichten_handen_vingers_pijn,"['gewrichten', 'handen', 'vingers', 'pijn', 'a...","['- Pijn in gewrichten/peesaanhechtingen.', 'I...",gewrichten - handen - vingers - pijn - arm - l...,0.000000,False,"Ik kreeg acuut een pijnlijke druk op de borst,...",M,msg_4,"Ik kreeg acuut een pijnlijke druk op de borst,...",sent_2
2,Het begon 1 uur na het avondeten.,52,52_eten_eet_voeding_gegeten,"['eten', 'eet', 'voeding', 'gegeten', 'eetlust...","['Wat mag ik wel/ niet eten?', 'ook na het ete...",eten - eet - voeding - gegeten - eetlust - dit...,0.776607,False,Het begon 1 uur na het avondeten.,M,msg_4,Het begon 1 uur na het avondeten.,sent_3
3,"Ik had al de hele dag migraine, had dus ook we...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.915956,False,"Ik had al de hele dag migraine, had dus ook we...",M,msg_4,"Ik had al de hele dag migraine, had dus ook we...",sent_4
4,"Ik werd heel erg misselijk, braakneigingen, du...",114,114_hoofdpijn_duizelig_duizeligheid_hoofd,"['hoofdpijn', 'duizelig', 'duizeligheid', 'hoo...",['Op dit moment merk ik door hoofdpijn en duiz...,hoofdpijn - duizelig - duizeligheid - hoofd - ...,0.000000,False,"Ik werd heel erg misselijk, braakneigingen, du...",M,msg_4,"Ik werd heel erg misselijk, braakneigingen, du...",sent_5


In [64]:
# print the unique count of "Topic" in merged_doc_sentence_df
print(f"Unique topics in merged_doc_sentence_df: {merged_doc_sentence_df['Topic'].nunique()}")

Unique topics in merged_doc_sentence_df: 157


In [81]:
# calculate the occurences of each "Mapped_Label" in "doc_info_df" grouped by "message_id"
mapped_label_counts_doc = merged_doc_sentence_df.groupby("message_id")["Mapped_Label"].value_counts().unstack(fill_value=0)
# check if "Mapped_Label" has empty values in mapped_label_counts_doc
print(mapped_label_counts_doc.isnull().sum())
# calculate the percentage of messages that have 0 "Mapped_Label" values that are "M"
percentage_no_m_doc = (mapped_label_counts_doc["M"] == 0).mean()
print(f"Percentage of messages with no 'M' label in doc_info_df: {percentage_no_m_doc:.2f}")

Mapped_Label
A    0
M    0
dtype: int64
Percentage of messages with no 'M' label in doc_info_df: 0.43


In [82]:
# in mapped_label_counts_doc, check how many unique "Topic" values has Mapped_Label value of "M", and how many unique "Topic" values has Mapped_Label value of "A"
topics_with_m = merged_doc_sentence_df[merged_doc_sentence_df["Mapped_Label"] == "M"]["Topic"].nunique()
topics_with_a = merged_doc_sentence_df[merged_doc_sentence_df["Mapped_Label"] == "A"]["Topic"].nunique()
print(f"Number of unique topics with 'M' label: {topics_with_m}")
print(f"Number of unique topics with 'A' label: {topics_with_a}")


Number of unique topics with 'M' label: 81
Number of unique topics with 'A' label: 76


In [69]:
# Topics that contain both M and A somewhere
topics_both = (
    merged_doc_sentence_df
    .groupby("Topic")["Mapped_Label"]
    .agg(lambda x: set(x.dropna()))
)

topics_both = topics_both[
    topics_both.apply(lambda labels: {"M", "A"}.issubset(labels))
]

print("Topics with both M and A:", len(topics_both))
print(topics_both)

Topics with both M and A: 0
Series([], Name: Mapped_Label, dtype: object)


In [70]:
topic_label_map["Most_Common_Label"].value_counts()

Most_Common_Label
M    81
A    73
Name: count, dtype: int64